In [ ]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Data

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/TSE2017 maldonaod/technical_debt_dataset.csv')
df['classification'].value_counts()

,count
classification,
WITHOUT_CLASSIFICATION,58204
DESIGN,2703
IMPLEMENTATION,757
DEFECT,472
TEST,85
DOCUMENTATION,54


In [ ]:
filtered_df = df[df['classification'].isin(['DESIGN', 'IMPLEMENTATION', 'WITHOUT_CLASSIFICATION'])].copy()
filtered_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 61664 entries, 13 to 62274
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   projectname     61664 non-null  object
 1   classification  61664 non-null  object
 2   commenttext     61664 non-null  object
dtypes: object(3)
memory usage: 1.9+ MB


In [ ]:
# Read the regular expression from the file
with open('/content/drive/MyDrive/TSE2017 maldonaod/technical_debt_dataset.csv', 'r') as file:
    java_regex_raw = file.read()

import re

# Function to preprocess the Java regex string into a Python-compatible regex
def convert_java_regex_to_python(java_regex_str):
    # Remove Java variable declaration prefix
    python_regex_pattern = re.sub(r'^\s*private static final String \w+ = \n?\s*"', '', java_regex_str, count=1)

    # Replace Java string concatenation ('"\n\t+ "') with Python regex OR operator ('|')
    python_regex_pattern = re.sub(r'"\s*\n?\s*\+\s*"', '|', python_regex_pattern)

    # Remove any remaining leading/trailing quotes and trailing semicolon
    python_regex_pattern = python_regex_pattern.strip('"').rstrip(';')

    # Escape common regex special characters that are likely meant as literals in this specific Java file context
    # We escape them only if they are not already preceded by a backslash
    python_regex_pattern = re.sub(r'(?<!\\)\(', r'\\(', python_regex_pattern)
    python_regex_pattern = re.sub(r'(?<!\\)\)', r'\\)', python_regex_pattern)
    python_regex_pattern = re.sub(r'(?<!\\)\{', r'\\{', python_regex_pattern)
    python_regex_pattern = re.sub(r'(?<!\\)\}', r'\\}', python_regex_pattern)

    return python_regex_pattern

# Convert the Java regex string
java_regex = convert_java_regex_to_python(java_regex_raw)

# Function to apply the regex cleaning
def clean_text_with_regex(text, regex):
    try:
        return re.sub(regex, '', text, flags=re.DOTALL)
    except Exception as e:
        return text

# Apply the cleaning to the 'commenttext' column
filtered_df['cleaned_commenttext'] = filtered_df['commenttext'].apply(lambda x: clean_text_with_regex(x, java_regex))

print("Original sample:")
print(filtered_df['commenttext'].head(2))
print("\nCleaned sample:")
print(filtered_df['cleaned_commenttext'].head(2))

Original sample:
13    /** XXX I really don't like this - the XML pro...
14    // JUnit 4 wraps solo tests this way. We can e...
Name: commenttext, dtype: object

Cleaned sample:
13    /** XXX I really don't like this - the XML pro...
14    // JUnit 4 wraps solo tests this way. We can e...
Name: cleaned_commenttext, dtype: object


In [ ]:
project = filtered_df['projectname'].unique()
results = {proj: {'DESIGN': 0, 'IMPLEMENTATION': 0, 'WITHOUT_CLASSIFICATION': 0} for proj in project}

for prj in project:
    vectorizer = TfidfVectorizer(max_features=2000,
                                ngram_range=(1, 2),
                                lowercase = True)
    model = LogisticRegression(max_iter=1000)
    train_df = filtered_df[filtered_df['projectname'] != prj]
    test_df = filtered_df[filtered_df['projectname'] == prj]

    X_train = vectorizer.fit_transform(train_df['cleaned_commenttext'])
    y_train = train_df['classification']

    X_test = vectorizer.transform(test_df['cleaned_commenttext'])
    y_test = test_df['classification']

    model.fit(X_train, y_train)

    predictions = model.predict(X_test)

    # Calculate F1 scores for each class
    f1_scores_per_class = f1_score(y_test, predictions, average=None, labels=model.classes_)

    # Map F1 scores to their respective classification names
    class_f1_map = dict(zip(model.classes_, f1_scores_per_class))

    # Store the F1 scores for 'DESIGN' and 'IMPLEMENTATION'
    if 'DESIGN' in class_f1_map: # Check if class was present in y_test
        results[prj]['DESIGN'] = class_f1_map['DESIGN']
    if 'IMPLEMENTATION' in class_f1_map: # Check if class was present in y_test
        results[prj]['IMPLEMENTATION'] = class_f1_map['IMPLEMENTATION']
    if 'WITHOUT_CLASSIFICATION' in class_f1_map:
        results[prj]['WITHOUT_CLASSIFICATION'] = class_f1_map['WITHOUT_CLASSIFICATION']


In [ ]:
if 'sum' in globals():
    del sum

print('F1 Scores per Project:')
for prj in results:
  design_f1 = results[prj].get('DESIGN', 0) # Use .get to handle cases where a class might not be present
  implementation_f1 = results[prj].get('IMPLEMENTATION', 0)
  without_f1 = results[prj].get('WITHOUT_CLASSIFICATION', 0)

  print(f"Project: {prj}")
  print(f"  DESIGN F1: {design_f1:.4f}")
  print(f"  IMPLEMENTATION F1: {implementation_f1:.4f}")
  print(f"  WITHOUT_CLASSIFICATION F1: {without_f1:.4f}")
  print("-" * 30)

# Calculate average F1 scores across all projects
avg_design_f1 = sum(results[prj].get('DESIGN', 0) for prj in results) / len(results)
avg_implementation_f1 = sum(results[prj].get('IMPLEMENTATION', 0) for prj in results) / len(results)
avg_without_f1 = sum(results[prj].get('WITHOUT_CLASSIFICATION', 0) for prj in results) / len(results)

print(f"\nAverage DESIGN F1 across projects: {avg_design_f1:.4f}")
print(f"Average IMPLEMENTATION F1 across projects: {avg_implementation_f1:.4f}")
print(f"Average WITHOUT_CLASSIFICATION F1 across projects: {avg_without_f1:.4f}")

F1 Scores per Project:
Project: apache-ant-1.7.0
  DESIGN F1: 0.3704
  IMPLEMENTATION F1: 0.0000
  WITHOUT_CLASSIFICATION F1: 0.9881
------------------------------
Project: apache-jmeter-2.10
  DESIGN F1: 0.6520
  IMPLEMENTATION F1: 0.0976
  WITHOUT_CLASSIFICATION F1: 0.9887
------------------------------
Project: argouml
  DESIGN F1: 0.6322
  IMPLEMENTATION F1: 0.2094
  WITHOUT_CLASSIFICATION F1: 0.9614
------------------------------
Project: columba-1.4-src
  DESIGN F1: 0.5253
  IMPLEMENTATION F1: 0.3636
  WITHOUT_CLASSIFICATION F1: 0.9940
------------------------------
Project: emf-2.4.1
  DESIGN F1: 0.4286
  IMPLEMENTATION F1: 0.0000
  WITHOUT_CLASSIFICATION F1: 0.9912
------------------------------
Project: hibernate-distribution-3.3.2.GA
  DESIGN F1: 0.6678
  IMPLEMENTATION F1: 0.1687
  WITHOUT_CLASSIFICATION F1: 0.9624
------------------------------
Project: jEdit-4.2
  DESIGN F1: 0.4195
  IMPLEMENTATION F1: 0.1333
  WITHOUT_CLASSIFICATION F1: 0.9918
----------------------------